# Step 2: Problem Framing, Data Loading & Target Definition

## 1. Problem Formulation: Credit Default Prediction
In retail banking and peer-to-peer (P2P) lending, an investor or institution lends a lump sum to an applicant, who promises to repay it via monthly installments with interest over a fixed horizon (typically 36 or 60 months).

The central credit risk question is: **"At the moment of origination, what is the probability that this applicant will default (charge off) before the loan matures?"**

### Target Definition Strategy:
- **`Class 0: Fully Paid`** (Good Borrower) -> Paid back principal plus interest in full.
- **`Class 1: Charged Off / Default`** (Bad Borrower) -> Severely delinquent, payments ceased, balance written off as an uncollectible loss.
- **`Drop: Current / In-Progress Loans`** -> Loans still actively being repaid have unknown future outcomes (**right-censoring**). Including them introduces severe label noise.

In [1]:
import sys
from pathlib import Path

# Ensure project root is in Python module path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.data_prep import load_raw_data, filter_closed_loans_and_define_target, inspect_data_health

print("Environment initialized successfully.")

Environment initialized successfully.


## 2. Load Raw Lending Club Data
We load the raw data stored in `data/raw/loan.csv`. The function `load_raw_data` provides options for chunking and reproducible sampling using `random_state=42`.

In [2]:
raw_data_path = PROJECT_ROOT / "data" / "raw" / "loan.csv"
df_raw = load_raw_data(raw_data_path)
print(f"Raw dataset dimensions: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

Loading raw loan data from: loan.csv...


Loaded raw records: 39,717 rows x 111 columns.
Raw dataset dimensions: 39,717 rows x 111 columns


## 3. Inspect Raw Loan Statuses
Before filtering, we examine all distinct values present in `loan_status`.

In [3]:
status_counts = df_raw["loan_status"].value_counts(dropna=False)
status_pcts = df_raw["loan_status"].value_counts(normalize=True, dropna=False) * 100

status_df = pd.DataFrame({"Count": status_counts, "Percentage (%)": status_pcts.round(2)})
display(status_df)

,Count,Percentage (%)
loan_status,,
Fully Paid,32950,82.96
Charged Off,5627,14.17
Current,1140,2.87


## 4. Filter Terminal Loans and Define Binary Target
We keep only completed loan contracts (`Fully Paid` and `Charged Off`), dropping active `Current` loans to prevent right-censorship label bias.

In [4]:
df_closed, summary = filter_closed_loans_and_define_target(df_raw)
print(f"Filtered closed loans: {df_closed.shape[0]:,} rows x {df_closed.shape[1]} columns")


--- Problem Framing & Target Definition Summary ---
Total raw loans evaluated:     39,717
In-progress loans dropped:     1,140 (right-censored)
Terminal loans retained:       38,577
  Class 0 (Fully Paid):        32,950 (85.41%)
  Class 1 (Default/Charged):   5,627 (14.59%)
Class imbalance ratio:         ~5.86:1
Filtered closed loans: 38,577 rows x 112 columns


## 5. Diagnostic Health Inspection
We inspect missing value prevalence across columns, memory footprint, and data types.

In [5]:
health = inspect_data_health(df_closed, target_col="target")

print("\n--- DATA HEALTH SUMMARY ---")
print(f"Total Rows:                 {health['num_rows']:,}")
print(f"Total Features:             {health['num_columns']}")
print(f"Memory Footprint:           {health['memory_usage_mb']} MB")
print(f"Columns 100% Missing:       {health['cols_100_missing_count']}")
print(f"Columns >50% Missing:       {health['cols_gt50_missing_count']}")
print(f"Columns <=50% Missing:      {health['cols_gt0_le50_missing_count']}")
print(f"Columns 100% Complete:      {health['cols_complete_count']}")
print(f"Data Type Counts:           {health['dtype_counts']}")

print("\n--- CLASS BALANCE ---")
for label, count in health["class_balance"]["counts"].items():
    name = "Class 0 (Fully Paid)" if label == 0 else "Class 1 (Default / Charged Off)"
    prop = health["class_balance"]["proportions"][label] * 100
    print(f"{name}: {count:,} ({prop:.2f}%)")


--- DATA HEALTH SUMMARY ---
Total Rows:                 38,577
Total Features:             112
Memory Footprint:           93.63 MB
Columns 100% Missing:       55
Columns >50% Missing:       2
Columns <=50% Missing:      11
Columns 100% Complete:      44
Data Type Counts:           {'float64': 74, 'object': 24, 'int64': 13, 'int32': 1}

--- CLASS BALANCE ---
Class 0 (Fully Paid): 32,950 (85.41%)
Class 1 (Default / Charged Off): 5,627 (14.59%)


## 6. Key Takeaways for Modeling
1. **Imbalance Ratio**: At ~14.6% default prevalence, the data exhibits moderate class imbalance (~5.86:1).
   - Accuracy is an invalid metric: predicting 0 everywhere yields 85.4% accuracy but captures 0% of bad loans.
   - We must prioritize **Recall, Precision, PR-AUC, and ROC-AUC**.
2. **Missing Columns**: 55 columns have 100% missing values (e.g. joint applicant fields). In Step 3, these will be pruned.
3. **Data Types**: Several numerical features (e.g. `term` with ' 36 months', `emp_length` with '10+ years') are stored as string objects and will require regex/numeric parsing.